# Multi-Seed Empirical Replication & Statistical Benchmark
### *Rigorous Empirical Search for Author Seeds across All 6 Deep Learning Models (Sharma et al., 2024)*
**Repository Branch:** `multiseed-replication` | **GitHub:** `Srr28/XAI-course-project`

---

### Motivation & Scientific Methodology
Sharma et al. (2024) omitted several operational hyperparameters, including random seeds and mini-batch sizes. Single-run evaluations are sensitive to initial random weights and train/val/test split partitions.

Building upon **LordKarsSama's multi-seed framework** (which was originally implemented only for 2D-CNN), this notebook generalizes the empirical search across **ALL SIX canonical models**:
1. **NSL-KDD**: DNN (`0.9930`), 1D-CNN (`0.9920`), 2D-CNN (`0.9940`)
2. **UNSW-NB15**: DNN (`0.8000`), 1D-CNN (`0.8000`), 2D-CNN (`0.8100`)

#### Methodological Rigor:
- **Seed Generation**: Deterministic 32-bit random seeds from master seeds (`20260908` for NSL, `20260909` for UNSW) matching LordKarsSama's generator.
- **Partitioning**: Exact stratified 60/15/25 re-partition per seed (`--split_mode resplit`), or canonical split.
- **Rounding-Aware Ranking**: Distance from the author's published rounding intervals ($[T - 0.5 \times 10^{-d}, T + 0.5 \times 10^{-d}]$).
- **Statistical Aggregations**: Sample Mean, Standard Deviation, and 95% Confidence Intervals.

## 1. Environment Setup, Repository Sync & Hardware Verification
Clones the `multiseed-replication` branch and verifies GPU availability.

In [ ]:
import os, sys, platform
from pathlib import Path

# Clone repository branch if running directly inside a fresh Google Colab session
if not os.path.exists('training'):
    print('Cloning multiseed-replication branch from GitHub fork...')
    !git clone -b multiseed-replication https://github.com/Srr28/XAI-course-project.git
    %cd XAI-course-project

print(f'Working Directory: {os.getcwd()}')
print(f'Python Version:    {platform.python_version()}')

# Verify GPU Acceleration
import tensorflow as tf
print(f'TensorFlow:        {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU Detected:      {gpus[0].name} (Hardware Acceleration Active!)')
else:
    print('WARNING: Running on CPU. For faster training, select Runtime -> Change runtime type -> T4 GPU.')

# Install requirements
!pip install -q -r requirements.txt

## 2. Verify Preprocessed Datasets
The preprocessed feature arrays are already version-controlled in the repository, ensuring zero data-download delays.

In [ ]:
import numpy as np

nsl_data = np.load('data/processed/nsl_kdd/nsl_kdd_processed.npz')
nsl_samples = len(nsl_data['y_train']) + len(nsl_data['y_val']) + len(nsl_data['y_test'])
print(f'[NSL-KDD] Total Samples: {nsl_samples:,}')
print(f'[NSL-KDD] Selected Features: {len(nsl_data["selected_features"])} (Input Grid: 6x6x1)')

unsw_data = np.load('data/processed/unsw_nb15/unsw_processed.npz')
unsw_samples = len(unsw_data['y_train']) + len(unsw_data['y_val']) + len(unsw_data['y_test'])
print(f'[UNSW-NB15] Total Samples: {unsw_samples:,}')
print(f'[UNSW-NB15] Selected Features: {len(unsw_data["selected_features"])} real + 11 zeros = 49 (Input Grid: 7x7x1)')

## 3. Fast Smoke-Test (2 Seeds, 2 Epochs)
Quick end-to-end check to verify seed generation, stratified splitting, model building, and reporting.

In [ ]:
!python run_multiseed.py --quick

## 4. Run Multi-Seed Empirical Sweep
Choose what to run:
- Sweep all 6 models across 5 seeds: `!python run_multiseed.py --all --num_seeds 5`
- Sweep a specific model (e.g. NSL 2D-CNN): `!python run_multiseed.py --exp_id NSL_SELECTED_2DCNN --num_seeds 10`
- Sweep all UNSW models: `!python run_multiseed.py --dataset unsw --num_seeds 5`

In [ ]:
# Run a 5-seed sweep on all 6 canonical models (20 epochs each)
!python run_multiseed.py --all --num_seeds 5 --epochs 20

## 5. Master Benchmark Results & Statistical Comparison
Displays the consolidated cross-model table and the comprehensive Markdown report.

In [ ]:
import os
import pandas as pd
from IPython.display import display, Markdown

comp_csv = 'results/multiseed/multiseed_paper_comparison.csv'
if os.path.exists(comp_csv):
    df = pd.read_csv(comp_csv)
    print('=== MASTER MULTI-SEED BENCHMARK TABLE ===')
    display(df)

report_md = 'results/multiseed/multiseed_report.md'
if os.path.exists(report_md):
    with open(report_md, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

## 6. Seed Distribution & Proximity Visualization
Generates empirical distribution plots for each evaluated model, comparing seed distributions against the published paper target with 95% Confidence Intervals.

In [ ]:
import glob
import json
import os
import matplotlib.pyplot as plt
import pandas as pd

model_dirs = sorted(glob.glob('results/multiseed/*/'))
models_found = [d for d in model_dirs if os.path.exists(os.path.join(d, 'seed_runs.csv'))]

if models_found:
    fig, axes = plt.subplots(len(models_found), 1, figsize=(10, 3.5 * len(models_found)))
    if len(models_found) == 1:
        axes = [axes]
    
    for ax, mdir in zip(axes, models_found):
        runs_df = pd.read_csv(os.path.join(mdir, 'seed_runs.csv'))
        with open(os.path.join(mdir, 'summary_statistics.json')) as f:
            stats = json.load(f)
        
        exp_id = stats['experiment_id']
        paper_acc = stats['paper_accuracy']
        mean_acc = stats['accuracy_mean']
        ci_low = stats['accuracy_ci95_low']
        ci_high = stats['accuracy_ci95_high']
        best_seed = stats['best_seed']['seed']
        best_acc = stats['best_seed']['test_accuracy']
        
        # Plot individual seed accuracies
        ax.scatter(runs_df['seed'].astype(str), runs_df['test_accuracy'], color='#2b5c8f', s=60, alpha=0.8, label='Seed Accuracy')
        
        # Highlight best matching seed
        ax.scatter([str(best_seed)], [best_acc], color='#d9534f', s=140, marker='*', zorder=5, label=f'Best Seed ({best_seed})')
        
        # Paper target horizontal line
        ax.axhline(paper_acc, color='#28a745', linestyle='--', linewidth=2, label=f'Paper Target ({paper_acc})')
        
        # Mean horizontal line
        ax.axhline(mean_acc, color='#333333', linestyle=':', linewidth=1.5, label=f'Mean ({mean_acc:.4f})')
        
        # 95% CI shaded band
        ax.axhspan(ci_low, ci_high, color='#2b5c8f', alpha=0.15, label='95% CI')
        
        ax.set_title(f'{exp_id} - Multi-Seed Accuracy Distribution vs. Paper Target', fontsize=12, fontweight='bold')
        ax.set_xlabel('Seed', fontsize=10)
        ax.set_ylabel('Test Accuracy', fontsize=10)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='best', fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print('No model run data found yet. Run Section 4 first.')

## 7. Download Results Archive
Package all generated CSVs, JSON summaries, and reports into a zip file for local inspection.

In [ ]:
!zip -r multiseed_results.zip results/multiseed/
print('\nResults archived to multiseed_results.zip')

# If in Google Colab, trigger browser download
try:
    from google.colab import files
    files.download('multiseed_results.zip')
except ImportError:
    print('Not running in Google Colab environment. File is saved at multiseed_results.zip')